# Feature Engineering


In [1]:
import numpy as np
import pandas as pd
import yfinance as yf
from market_ml.data_loader import DataLoader
import matplotlib.pyplot as plt
import plotly.express as px
from market_ml.config import * 
from market_ml.EDA import * 
from market_ml.feature_engineering import *
from market_ml.models import *

from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LinearRegression
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import *
from sklearn.model_selection import GridSearchCV, TimeSeriesSplit


# Data Loading 


In [2]:
stock_dl = DataLoader(tickers=TICKERS, period="10y", interval="1d", file_name='stock_data.parquet')
benchmark_dl = DataLoader(tickers=[BENCHMARK_TICKERS], period="10y", interval="1d", file_name='benchmark_data.parquet')

market_data = stock_dl.load_data()
benchmark_data = benchmark_dl.load_data()
sectors = market_data["Sector"].unique()

benchmark_data

Data loaded for ['AAPL', 'MSFT', 'NVDA', 'META', 'GOOGL', 'JPM', 'GS', 'BAC', 'MS', 'BLK', 'JNJ', 'UNH', 'ABBV', 'PFE', 'MRK', 'XOM', 'CVX', 'COP', 'SLB', 'EOG', 'AMZN', 'COST', 'WMT', 'HD', 'MCD']
Data loaded for [['SPY', 'QQQ']]


Price,Date,Ticker,Sector,Close,High,Low,Open,Volume
0,2016-09-29,QQQ,NaN,110.121559,110.943924,109.644969,110.738333,27207500
1,2016-09-29,SPY,NaN,183.335754,185.206007,182.789198,184.804628,128070600
2,2016-09-30,QQQ,NaN,110.943939,111.271013,110.299131,110.411274,25543200
3,2016-09-30,SPY,NaN,184.719223,185.419493,183.916465,184.164118,117202900
4,2016-10-03,QQQ,NaN,110.785042,110.925211,110.355165,110.803729,19216400
...,...,...,...,...,...,...,...,...
5021,2026-09-25,SPY,NaN,771.349976,772.280029,766.289978,768.780029,36666700
5022,2026-09-28,QQQ,NaN,736.530029,741.419983,731.630005,740.460022,41716200
5023,2026-09-28,SPY,NaN,765.609985,769.539978,763.719971,768.349976,42296700
5024,2026-09-29,QQQ,NaN,737.229980,740.579895,736.000000,740.169983,9703756


In [3]:
#just testing with one stock for now 

test_ticker = ['AAPL', 'NVDA']
test_data = market_data[market_data["Ticker"].isin(test_ticker)].copy()

test_data

Price,Date,Ticker,Sector,Close,High,Low,Open,Volume
0,2016-09-29,AAPL,Technology,25.653965,26.024437,25.567065,25.878078,143548000
19,2016-09-29,NVDA,Technology,1.653514,1.658666,1.620394,1.635605,336684000
25,2016-09-30,AAPL,Technology,25.852909,25.926088,25.567052,25.717984,145516400
44,2016-09-30,NVDA,Technology,1.680990,1.697427,1.652778,1.662591,429932000
50,2016-10-03,AAPL,Technology,25.731701,25.852906,25.676817,25.775152,86807200
...,...,...,...,...,...,...,...,...
62769,2026-09-25,NVDA,Technology,225.070007,226.940002,223.130005,225.130005,89947700
62775,2026-09-28,AAPL,Technology,338.399994,342.989990,338.040009,340.369995,32724700
62794,2026-09-28,NVDA,Technology,228.860001,233.210007,228.039993,229.750000,141544300
62800,2026-09-29,AAPL,Technology,332.345001,337.059998,331.510010,337.059998,12025955


In [4]:
engineer = FeatureEngineer(test_data, benchmark_data, filename="test_stock_data_with_features.parquet")
data  = engineer.load_features()

data.shape

Data loaded for Price       Date Ticker      Sector       Close        High         Low  \
0     2016-09-29   AAPL  Technology   25.653965   26.024437   25.567065   
19    2016-09-29   NVDA  Technology    1.653514    1.658666    1.620394   
25    2016-09-30   AAPL  Technology   25.852909   25.926088   25.567052   
44    2016-09-30   NVDA  Technology    1.680990    1.697427    1.652778   
50    2016-10-03   AAPL  Technology   25.731701   25.852906   25.676817   
...          ...    ...         ...         ...         ...         ...   
62769 2026-09-25   NVDA  Technology  225.070007  226.940002  223.130005   
62775 2026-09-28   AAPL  Technology  338.399994  342.989990  338.040009   
62794 2026-09-28   NVDA  Technology  228.860001  233.210007  228.039993   
62800 2026-09-29   AAPL  Technology  332.345001  337.059998  331.510010   
62819 2026-09-29   NVDA  Technology  230.899994  232.820007  229.220001   

Price        Open     Volume  
0       25.878078  143548000  
19       1.635605  33

(4568, 27)

In [5]:
print(data.shape)

print(data[["Date", "Ticker"]].head())

print(data.sort_values(["Date", "Ticker"]).reset_index(drop=True))



(4568, 27)
Price       Date Ticker
98    2016-12-08   AAPL
99    2016-12-08   NVDA
100   2016-12-09   AAPL
101   2016-12-09   NVDA
102   2016-12-12   AAPL
Price       Date Ticker      Sector       Close        High         Low  \
0     2016-12-08   AAPL  Technology   25.771877   25.843133   25.422489   
1     2016-12-08   NVDA  Technology    2.296763    2.373912    2.286199   
2     2016-12-09   AAPL  Technology   26.192514   26.364909   25.815545   
3     2016-12-09   NVDA  Technology    2.255978    2.316174    2.228706   
4     2016-12-12   AAPL  Technology   26.043110   26.433871   25.856922   
...          ...    ...         ...         ...         ...         ...   
4563  2026-01-07   NVDA  Technology  188.668518  190.923237  186.124468   
4564  2026-01-08   AAPL  Technology  258.336823  258.586144  255.005878   
4565  2026-01-08   NVDA  Technology  184.608002  189.107482  183.281120   
4566  2026-01-09   AAPL  Technology  258.665924  259.503640  255.524481   
4567  2026-01-09   N

In [6]:
print("Start:", data["Date"].min())
print("End:  ", data["Date"].max())

Start: 2016-12-08 00:00:00
End:   2026-01-09 00:00:00


# Splitting Data into Train and Test 

In [7]:
data = data.drop(columns=[col for col in data.columns if "Target" in col]) #target was created for categorical, we don't want these in the regression
train_data, test_data, split_date = split_chronological_data(data)

data

Price,Date,Ticker,Sector,Close,High,Low,Open,Volume,QQQ_Return,SPY_Return,...,SMA_20D,SMA_50D,Price_to_SMA_20D,Price_to_SMA_50D,Volume_Average_20D,Relative_Volume_20D,Future_Return_5D,Future_Return_10D,Future_Return_20D,Future_Return_60D
98,2016-12-08,AAPL,Technology,25.771877,25.843133,25.422489,25.482253,108273200,0.001774,0.002449,...,25.293078,25.814255,1.018930,0.998358,131367840.0,0.824199,0.033001,0.037192,0.061274,0.245118
99,2016-12-08,NVDA,Technology,2.296763,2.373912,2.286199,2.338778,381544000,0.001774,0.002449,...,2.218632,1.894732,1.035216,1.212184,673159800.0,0.566796,0.055948,0.145807,0.147625,0.055674
100,2016-12-09,AAPL,Technology,26.192514,26.364909,25.815545,25.815545,137610400,0.007843,0.006040,...,25.363874,25.825026,1.032670,1.014230,126821460.0,1.085072,0.017727,0.022554,0.045283,0.222301
101,2016-12-09,NVDA,Technology,2.255978,2.316174,2.228706,2.314454,382312000,0.007843,0.006040,...,2.248302,1.906782,1.003414,1.183134,648811400.0,0.589250,0.093553,0.195600,0.159552,0.074541
102,2016-12-12,AAPL,Technology,26.043110,26.433871,25.856922,26.040811,105497600,-0.004519,-0.001147,...,25.419845,25.828830,1.024519,1.008296,125267560.0,0.842178,0.029479,0.034952,0.056929,0.233391
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
4661,2026-01-07,NVDA,Technology,188.668518,190.923237,186.124468,188.129785,153543200,0.000963,-0.003224,...,183.191335,186.384763,1.029899,1.012253,162012720.0,0.947723,-0.031569,-0.022579,-0.091111,-0.060602
4662,2026-01-08,AAPL,Technology,258.336823,258.586144,255.005878,256.322286,50419300,-0.005689,-0.000102,...,270.800395,272.026132,0.953975,0.949676,42681970.0,1.181279,-0.003204,-0.042465,0.073657,-0.020471
4663,2026-01-08,NVDA,Technology,184.608002,189.107482,183.281120,188.668507,172457000,-0.005689,-0.000102,...,183.194825,186.256276,1.007714,0.991151,163399585.0,1.055431,0.010863,0.014213,0.002000,-0.037453
4664,2026-01-09,AAPL,Technology,258.665924,259.503640,255.524481,258.376703,39997000,0.009960,0.006613,...,269.832530,271.839251,0.958617,0.951540,43029905.0,0.929516,-0.014805,-0.015268,0.059787,-0.000878


In [8]:
data.columns

Index(['Date', 'Ticker', 'Sector', 'Close', 'High', 'Low', 'Open', 'Volume',
       'QQQ_Return', 'SPY_Return', 'Daily_Return', 'Return_5D',
       'Volatility_20D', 'SMA_20D', 'SMA_50D', 'Price_to_SMA_20D',
       'Price_to_SMA_50D', 'Volume_Average_20D', 'Relative_Volume_20D',
       'Future_Return_5D', 'Future_Return_10D', 'Future_Return_20D',
       'Future_Return_60D'],
      dtype='str', name='Price')

In [9]:
features = list(train_data.columns)[3:-1] #not including the date column
target = list(train_data)[-1] #target is the 5 day return. 
features

['Close',
 'High',
 'Low',
 'Open',
 'Volume',
 'QQQ_Return',
 'SPY_Return',
 'Daily_Return',
 'Return_5D',
 'Volatility_20D',
 'SMA_20D',
 'SMA_50D',
 'Price_to_SMA_20D',
 'Price_to_SMA_50D',
 'Volume_Average_20D',
 'Relative_Volume_20D',
 'Future_Return_5D',
 'Future_Return_10D',
 'Future_Return_20D']

In [10]:
x_train, y_train = train_data[features], train_data[target]
x_test, y_test = test_data[features], test_data[target]

x_test.columns

Index(['Close', 'High', 'Low', 'Open', 'Volume', 'QQQ_Return', 'SPY_Return',
       'Daily_Return', 'Return_5D', 'Volatility_20D', 'SMA_20D', 'SMA_50D',
       'Price_to_SMA_20D', 'Price_to_SMA_50D', 'Volume_Average_20D',
       'Relative_Volume_20D', 'Future_Return_5D', 'Future_Return_10D',
       'Future_Return_20D'],
      dtype='str', name='Price')

# Baseline Model

want random model to compare to. 

In [11]:
model_comparison_df = pd.DataFrame(columns=["Model", "MAE", "RMSE", "R2"]) 

In [12]:
baseline_prediction = y_train.mean()
baseline_y_pred = [baseline_prediction] * len(y_test)
print(f"Baseline prediction: {np.round(baseline_prediction,4)}")

Baseline prediction: 0.1144


In [13]:
model_comparison_df = pd.DataFrame(columns=["Model", "MAE", "RMSE", "R2"]) 

baseline_mae = mean_absolute_error(y_test, baseline_y_pred)
baseline_rmse = root_mean_squared_error(y_test, baseline_y_pred)
baseline_r2 = r2_score(y_test, baseline_y_pred)

print(f"Baseline Mean absolute error is {np.round(baseline_mae,4)}")
print(f"Baseline Root mean squared error is {np.round(baseline_rmse,4)}")
print(f"Baseline R-squared is {np.round(baseline_r2,4)}")

baseline_metrics = ["Baseline Metrics", np.round(baseline_mae,4), np.round(baseline_rmse,4), np.round(baseline_r2,4)]

if baseline_metrics[0] not in model_comparison_df.columns:
    model_comparison_df.loc[len(model_comparison_df)] = baseline_metrics
model_comparison_df


Baseline Mean absolute error is 0.1447
Baseline Root mean squared error is 0.1802
Baseline R-squared is -0.0262


,Model,MAE,RMSE,R2
0,Baseline Metrics,0.1447,0.1802,-0.0262


# Linear Regression

Not using standard scaler initially  

In [14]:
linearModel = LinearRegression()

linearModel.fit(x_train, y_train)

y_pred = linearModel.predict(x_test)

In [15]:
MAE = mean_absolute_error(y_test, y_pred) #average size of the prediction error
RMSE = root_mean_squared_error(y_test, y_pred) #how far the predictions are from the actual values, penalises large errors.
R2 = r2_score(y_test, y_pred) #how much variation in the target variable the model predicts (1.0 is perfect).

print(f"Linear Regression Mean absolute error is {np.round(MAE,4)}")
print(f"Linear Regression Root mean squared error is {np.round(RMSE,4)}")
print(f"Linear Regression R-squared is {np.round(R2,4)}")

linear_regression_metrics = ["Linear Regression", np.round(MAE,4), np.round(RMSE,4), np.round(R2, 4)]

if linear_regression_metrics[0] not in model_comparison_df.columns:
    model_comparison_df.loc[len(model_comparison_df)] = linear_regression_metrics

model_comparison_df

Linear Regression Mean absolute error is 0.1361
Linear Regression Root mean squared error is 0.1721
Linear Regression R-squared is 0.0635


,Model,MAE,RMSE,R2
0,Baseline Metrics,0.1447,0.1802,-0.0262
1,Linear Regression,0.1361,0.1721,0.0635


Looks like our initial linear regression is pretty close to baseline. 

# Random Forest

In [16]:
ntrees = 100
max_depth = 5 #default is None which I don't want to use at it'll overfit
random_state = 42 #for reproducability 
njobs = -1 #allowing it to use all available cores on my computer 
rf_model = RandomForestRegressor(n_estimators=ntrees, max_depth=max_depth, random_state=random_state, n_jobs=njobs)

In [17]:
rf_model.fit(x_train, y_train)
y_pred = rf_model.predict(x_test)

In [18]:
MAE = mean_absolute_error(y_test, y_pred)
RMSE = root_mean_squared_error(y_test, y_pred)
R2 = r2_score(y_test, y_pred)

print(f"Mean absolute error is {np.round(MAE,4)}")
print(f"Root mean squared error is {np.round(RMSE,4)}")
print(f"R-squared is {np.round(R2,4)}")

random_forest_metrics = ["Random Forest", np.round(MAE,4), np.round(RMSE,4), np.round(R2, 4)]

if random_forest_metrics[0] not in model_comparison_df.columns:
    model_comparison_df.loc[len(model_comparison_df)] = random_forest_metrics
    model_comparison_df

model_comparison_df

Mean absolute error is 0.1285
Root mean squared error is 0.164
R-squared is 0.1495


,Model,MAE,RMSE,R2
0,Baseline Metrics,0.1447,0.1802,-0.0262
1,Linear Regression,0.1361,0.1721,0.0635
2,Random Forest,0.1285,0.1640,0.1495


# Random Forest Hyperparameter Tuning

In [19]:
rf_Model = RandomForestRegressor(random_state=42)

cv = TimeSeriesSplit(n_splits=5)

param_grid = {
    "n_estimators":[50, 100, 200, 300, 400],
    "max_depth": [3, 5, 10, 15, 20],
    "min_samples_leaf": [1, 2, 5],
}

grid_search = GridSearchCV(rf_Model,
    param_grid=param_grid,
    cv=cv,
    scoring="neg_mean_absolute_error",
    n_jobs=-1
)



In [20]:
grid_search.fit(x_train, y_train)

print(grid_search.best_params_)
print(grid_search.best_score_)

{'max_depth': 3, 'min_samples_leaf': 1, 'n_estimators': 400}
-0.184331188135113


In [21]:
best_parameters = grid_search.best_params_

rf_Model = RandomForestRegressor(n_estimators=best_parameters["n_estimators"], max_depth=best_parameters["max_depth"], min_samples_leaf=best_parameters["min_samples_leaf"], random_state=42)

rf_Model.fit(x_train, y_train)
y_pred = rf_Model.predict(x_test)




In [22]:
MAE = mean_absolute_error(y_test, y_pred)
RMSE = root_mean_squared_error(y_test, y_pred)
R2 = r2_score(y_test, y_pred)

print(f"Tuned Random Forest Mean absolute error is {np.round(MAE,4)}")
print(f"Tuned Random Forest Root mean squared error is {np.round(RMSE,4)}")
print(f"Tuned Random R-squared is {np.round(R2,4)}")

tuned_random_forest_metrics = ["Tuned Random Forest", np.round(MAE,4), np.round(RMSE, 4), np.round(R2, 4)]

if tuned_random_forest_metrics[0] not in model_comparison_df.columns:
    model_comparison_df.loc[len(model_comparison_df)] = tuned_random_forest_metrics
    model_comparison_df

model_comparison_df

Tuned Random Forest Mean absolute error is 0.1186
Tuned Random Forest Root mean squared error is 0.1545
Tuned Random R-squared is 0.2452


,Model,MAE,RMSE,R2
0,Baseline Metrics,0.1447,0.1802,-0.0262
1,Linear Regression,0.1361,0.1721,0.0635
2,Random Forest,0.1285,0.1640,0.1495
3,Tuned Random Forest,0.1186,0.1545,0.2452


Random forest is improving with additional features, now to see if this improves with the full 25 stock universe. 